# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [15]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Aug 25 11:30:12 PM 2026"

In [16]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,760469,40.7,1473300,78.7,1473300,78.7
Vcells,1497637,11.5,127274484,971.1,149262973,1138.8


In [17]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia

In [18]:
PARAM <- list()
PARAM$semilla_primigenia <- 346321

# parametros  arbol
# entreno cada arbol con solo 50% de las variables variables
#  por ahora, es fijo
PARAM$feature_fraction <- 0.5

PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 200
PARAM$rpart$minbucket <- 66
PARAM$rpart$maxdepth <- 12

# voy a generar 512 arboles,
#  a mas arboles mas tiempo de proceso y MEJOR MODELO,
#  pero ganancias marginales
PARAM$num_trees_max <- 32

In [ ]:
PARAM

$semilla_primigenia
[1] 346321

$feature_fraction
[1] 0.5

$rpart
$rpart$cp
[1] -1

$rpart$minsplit
[1] 1000

$rpart$minbucket
[1] 100

$rpart$maxdepth
[1] 8


$num_trees_max
[1] 32

In [19]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp433"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [20]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [21]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [22]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [23]:
# que tamanos de ensemble grabo a disco
grabar <- c(32)
grabar

[1] 32

In [24]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
# aqui se va acumulando la probabilidad del ensemble
tb_prediccion[, prob_acumulada := 0]

In [25]:
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [26]:
# archivo donde se guarda el progreso, para poder retomar si se corta la conexion
archivo_checkpoint <- "checkpoint.RDATA"

if (file.exists(archivo_checkpoint)) {
  # ya existe un checkpoint de una corrida anterior interrumpida
  load(archivo_checkpoint) # trae tb_prediccion, arbolito_ultimo y .Random.seed
  arbolito_inicio <- arbolito_ultimo + 1
  message("Retomando desde el checkpoint,  ultimo arbolito completado= ", arbolito_ultimo)
} else {
  arbolito_inicio <- 1
}

if (arbolito_inicio > PARAM$num_trees_max) {

  message("El ensemble ya estaba completo ( ", PARAM$num_trees_max, " arboles ),  nada para hacer")

} else {

  for (arbolito in arbolito_inicio:PARAM$num_trees_max) {
    message( arbolito, " ")
    qty_campos_a_utilizar <- as.integer(length(campos_buenos)
      * PARAM$feature_fraction)

    # elijo los campos al azar
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)

    # paso de un vector a un string con los elementos
    # separados por un signo de "+"
    # este hace falta para la formula
    campos_random <- paste(campos_random, collapse= " + ")

    # armo la formula para rpart
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    # genero el arbol de decision
    modelo <- rpart(formulita,
      data= dtrain,
      xval= 0,
      control= PARAM$rpart
    )

    # aplico el modelo a los datos que no tienen clase
    prediccion <- predict(modelo, dfuture, type= "prob")

    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    # grabo el checkpoint en cada arbolito ( tb_prediccion + semilla + ultimo arbolito completado )
    # para poder retomar exactamente desde aca si se corta la conexion
    arbolito_ultimo <- arbolito
    save(tb_prediccion, arbolito_ultimo, .Random.seed, file= archivo_checkpoint)

    if (arbolito %in% grabar) {
      umbral_corte <- (1 / 40) * arbolito
      tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

      archivo_kaggle <- paste0(
          "KA333_",
          sprintf("%.3d", arbolito), # para que tenga ceros adelante
          ".csv"
        )

      # grabo el archivo
      fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
        file= archivo_kaggle,
        sep= ","
      )

      # subida a Kaggle
      comando <- "kaggle competitions submit"
      competencia <- "-c utn-2026-inicial"
      arch <- paste( "-f", archivo_kaggle)

      mensaje <- paste0("-m 'cp=", PARAM$rpart$cp, "  minsplit=", PARAM$rpart$minsplit, "  minbucket=", PARAM$rpart$minbucket, " maxdepth=", PARAM$rpart$maxdepth, "'" )
      linea <- paste( comando, competencia, arch, mensaje)
      salida <- system(linea, intern=TRUE)
      cat(salida)
    }
  }

}


1 

2 

3 

4 

5 

6 

7 

8 

9 

10 

11 

12 

13 

14 

15 

16 

17 

18 

19 

20 

21 

22 

23 

24 

25 

26 

27 

28 

29 

30 

31 

32 



In [ ]:
qty_campos_a_utilizar <- as.integer(length(campos_buenos)
      * PARAM$feature_fraction)

# elijo los campos al azar
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)

    # paso de un vector a un string con los elementos
    # separados por un signo de "+"
    # este hace falta para la formula
    campos_random <- paste(campos_random, collapse= " + ")

    # armo la formula para rpart
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    # genero el arbol de decision
    modelo <- rpart(formulita,
      data= dtrain,
      xval= 0,
      control= PARAM$rpart
    )

    # aplico el modelo a los datos que no tienen clase
    prediccion <- predict(modelo, dfuture, type= "prob")

    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    # grabo el checkpoint en cada arbolito ( tb_prediccion + semilla + ultimo arbolito completado )
    # para poder retomar exactamente desde aca si se corta la conexion
    arbolito_ultimo <- arbolito

ERROR: Error: object 'tb_prediccion' not found


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")



---

